# Stockholm POIs → 11 stadsdelsområden (fixed)

Reads `stockholm_poi_enriched_analysis.csv`, downloads the `Stadsdelsnämnder_2025` ArcGIS boundary layer, assigns POIs to administrative areas by location, and exports an interactive HTML map. **Run cells in order**. This notebook does not invent district assignments: unmatched POIs remain blank. Requires internet for initial boundary download and map tiles. The boundaries are cached locally.

In [1]:
%pip install pandas geopandas shapely requests folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Kim\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
import json, re, unicodedata
import pandas as pd
import geopandas as gpd
import requests

DATA = Path('stockholm_poi_enriched_analysis.csv')
BOUNDARY = Path('stockholm_stadsdelsomraden_2025.geojson')
OUTPUT = Path('stockholm_poi_with_11_areas.csv')
HTML_FILE = Path('stockholm_poi_11_areas_map.html')
SERVICE = 'https://services2.arcgis.com/uM75dQp2RlOXWzhF/ArcGIS/rest/services/Stadsdelsnämnder_2025/FeatureServer'

if not DATA.exists():
    raise FileNotFoundError(f'Place {DATA.name} in the same folder as this notebook. Current directory: {Path.cwd()}')
pois = pd.read_csv(DATA, low_memory=False)
print(f'Loaded {len(pois):,} POIs from {DATA.resolve()}')
print('Coordinate columns available:', {'latitude','longitude'}.issubset(pois.columns))


Loaded 4,234 POIs from C:\Users\Kim\software engineering\year 5\LP1\4DT911\4dt911-vis-project-research\notebooks\data_preprocessing\tourism\stockholm_poi_enriched_analysis.csv
Coordinate columns available: True


In [3]:

# Download and cache polygons from the ArcGIS feature service.
# If blocked on your network, export the service layer as GeoJSON manually
# and save it as stockholm_stadsdelsomraden_2025.geojson.
def get_json(url, params):
    try:
        r = requests.get(url, params=params, timeout=60, headers={'User-Agent':'StockholmUniversityPOIStudy/1.0'})
        r.raise_for_status()
        obj = r.json()
        if 'error' in obj:
            raise RuntimeError(str(obj['error']))
        return obj
    except Exception as exc:
        raise RuntimeError(f'Failed to fetch {url}: {exc}. Download the 2025 boundary as GeoJSON manually and save it as {BOUNDARY.name}.') from exc

if not BOUNDARY.exists():
    info = get_json(SERVICE, {'f':'json'})
    layers = info.get('layers', [])
    if not layers:
        raise RuntimeError('The ArcGIS service returned no layers. Check the service in your browser.')
    all_features=[]
    for layer in layers:
        layer_url=f"{SERVICE}/{layer['id']}/query"
        data=get_json(layer_url, {'f':'geojson','where':'1=1','outFields':'*','returnGeometry':'true','outSR':'4326'})
        features=data.get('features', [])
        if features:
            all_features=features
            break
    if not all_features:
        raise RuntimeError('No geographic features were returned by ArcGIS. Manually export the service layer to GeoJSON.')
    BOUNDARY.write_text(json.dumps({'type':'FeatureCollection','features':all_features},ensure_ascii=False),encoding='utf-8')
    print('Downloaded:', BOUNDARY)
else:
    print('Using cached boundary file:', BOUNDARY)

raw=gpd.read_file(BOUNDARY).to_crs('EPSG:4326')
print('Boundary records:',len(raw),' | Fields:', list(raw.columns))


Using cached boundary file: stockholm_stadsdelsomraden_2025.geojson
Boundary records: 11  | Fields: ['FID', 'Namn', 'Typ', 'Shape__Area', 'Shape__Length', 'geometry']


In [4]:

# Identify the 11 boroughs, allowing official Swedish names and suffixes.
EXPECTED = ['Bromma', 'Enskede-Årsta-Vantör', 'Farsta', 'Hägersten-Älvsjö',
            'Hässelby-Vällingby', 'Järva', 'Kungsholmen', 'Norra innerstaden',
            'Skarpnäck', 'Skärholmen', 'Södermalm']

def simplify(s):
    s = unicodedata.normalize('NFKD', str(s or '').casefold())
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.replace('–','-').replace('—','-')
    return re.sub(r'[^a-z0-9]+','',s)

NAME_KEYS={simplify(v):v for v in EXPECTED}
ALIAS={simplify('Norra innerstadens'): 'Norra innerstaden',
       simplify('Enskede-Årsta-Vantörs'): 'Enskede-Årsta-Vantör',
       simplify('Hässelby Vällingby'): 'Hässelby-Vällingby'}

def canonical(value):
    key=simplify(value)
    if key in NAME_KEYS: return NAME_KEYS[key]
    if key in ALIAS: return ALIAS[key]
    # Names such as 'Södermalms stadsdelsnämnd' or 'Norra innerstadens stadsdelsområde'
    for suffix in ['stadsdelsnamnd','stadsdelsomrade','stadsdelsforvaltning']:
        if key.endswith(suffix):
            base=key[:-len(suffix)]
            if base in NAME_KEYS: return NAME_KEYS[base]
            if base in ALIAS: return ALIAS[base]
            if base.endswith('s') and base[:-1] in NAME_KEYS: return NAME_KEYS[base[:-1]]
    return None

candidates=[]
for col in raw.columns:
    if col=='geometry': continue
    found=raw[col].map(canonical)
    candidates.append((found.nunique(), found.notna().sum(),col))
print('Candidate name fields:', sorted(candidates,reverse=True)[:6])
name_field=max(candidates,default=(0,0,None))[2]
if name_field is None or max(candidates)[0] < 8:
    raise ValueError('Could not identify borough names automatically. Inspect raw.head() and boundary column values; no guesses were made.')

boroughs=raw[['geometry']].copy()
boroughs['stadsdelsomrade']=raw[name_field].map(canonical)
boroughs=boroughs.dropna(subset=['stadsdelsomrade'])
boroughs=boroughs.dissolve(by='stadsdelsomrade',as_index=False)
print('Recognized boroughs:', sorted(boroughs.stadsdelsomrade.unique()))
missing=set(EXPECTED)-set(boroughs.stadsdelsomrade)
if missing:
    raise ValueError(f'Missing {len(missing)} borough names: {sorted(missing)}. Check the boundary layer and name field: {name_field}')


Candidate name fields: [(7, np.int64(7), 'Namn'), (0, np.int64(0), 'Typ'), (0, np.int64(0), 'Shape__Length'), (0, np.int64(0), 'Shape__Area'), (0, np.int64(0), 'FID')]


ValueError: Could not identify borough names automatically. Inspect raw.head() and boundary column values; no guesses were made.

In [ ]:

# Assign all POIs using a spatial join (no invented assignment for unmatched points).
base=pois.copy().reset_index(drop=True)
base['latitude']=pd.to_numeric(base.latitude,errors='coerce')
base['longitude']=pd.to_numeric(base.longitude,errors='coerce')
base['_row_id']=range(len(base))
valid=base.latitude.notna() & base.longitude.notna()
points=gpd.GeoDataFrame(base.loc[valid].copy(),geometry=gpd.points_from_xy(base.loc[valid,'longitude'],base.loc[valid,'latitude']),crs='EPSG:4326')
matched=gpd.sjoin(points,boroughs,how='left',predicate='intersects')
duplicated_rows=int(matched['_row_id'].duplicated().sum())
matched=matched.sort_values(['_row_id','stadsdelsomrade'],na_position='last').drop_duplicates('_row_id')
lookup=matched.set_index('_row_id')['stadsdelsomrade']
base['stadsdelsomrade']=base['_row_id'].map(lookup)
base['stadsdelsomrade_match_status']=base.stadsdelsomrade.notna().map({True:'matched',False:'unmatched'})
base['stadsdelsomrade_boundary_source']='ArcGIS Stadsdelsnämnder_2025'
result=base.drop(columns=['_row_id'])
result.to_csv(OUTPUT,index=False,encoding='utf-8-sig')
print(f'Saved: {OUTPUT.resolve()}')
print(f'Assigned: {result.stadsdelsomrade.notna().sum():,}/{len(result):,}; boundary overlaps: {duplicated_rows}')
print(result.stadsdelsomrade.value_counts(dropna=False).to_string())
display(result[['name','analysis_group','stadsdelsomrade']].head(10))


In [ ]:

# Interactive map: hover for name/type/borough and click for more attributes.
import folium
from folium.plugins import MarkerCluster
from html import escape
from IPython.display import display

m=folium.Map(location=[59.3293,18.0686],zoom_start=11,tiles=None,control_scale=True)
folium.TileLayer(tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                 attr='Tiles © Esri', name='Esri streets').add_to(m)
folium.GeoJson(boroughs.to_json(),name='11 stadsdelsområden',style_function=lambda f: {'color':'#9d4242','weight':2,'fillOpacity':0.04},
               tooltip=folium.GeoJsonTooltip(fields=['stadsdelsomrade'],aliases=['Area:'])).add_to(m)
colors={'food_drink':'#d4843c','nightlife':'#9454a3','visitor_attraction':'#1769aa',
        'culture_heritage':'#ba5457','parks_recreation':'#3e9365','culture_entertainment':'#a48c30'}

def safe(v):
    return escape(str(v)) if pd.notna(v) else '—'

for group,sub in result.loc[valid].groupby('analysis_group',dropna=False):
    layer=folium.FeatureGroup(name=f'{group} ({len(sub)})',show=True)
    cluster=MarkerCluster().add_to(layer)
    for row in sub.itertuples(index=False):
        r=row._asdict()
        name=safe(r.get('name')); area=safe(r.get('stadsdelsomrade'))
        typ=safe(r.get('primary_subcategory'))
        tooltip=f'<b>{name}</b><br>Type: {typ}<br>Area: {area}'
        fields=['opening_hours','best_website','cuisine','wheelchair','all_pois_within_500m','osm_url']
        details=''.join(f'<b>{escape(k)}:</b> {safe(r[k])}<br>' for k in fields if pd.notna(r.get(k)))
        popup=f'<div style="min-width:220px"><b>{name}</b><br><b>Area:</b> {area}<br><b>Type:</b> {typ}<br>{details}</div>'
        folium.CircleMarker([float(r['latitude']),float(r['longitude'])],radius=5,color=colors.get(group,'#777'),
          fill=True,fill_opacity=.8,tooltip=folium.Tooltip(tooltip,sticky=True),popup=folium.Popup(popup,max_width=370)).add_to(cluster)
    layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(HTML_FILE)
print('Saved HTML:',HTML_FILE.resolve())
display(m)
